# Continuous Sign Language Recognition on all 75 keypoints (csrl-faithful)

A faithful reimplementation of the `csrl` framework (TSSI → dilated TCN → Temporal
Attention → BiLSTM → CTC, with an auxiliary SR-CTC head), **extended to all 75 MediaPipe
keypoints** (33 body + 21 + 21 hands) on the **MSKA-SLR** data format.

Aligned with the reference csrl setup:
- **gloss merge** (SequenceMatcher + hyphen normalisation) → reduced vocabulary, easier CTC;
- **Optuna hyperparameters** in `CFG`;
- **six augmentations** (jitter, temporal warp, L/R flip, scaling, frame-drop, confidence-drop);
- **two-phase training**: Phase 1 with a frozen backbone (head only), Phase 2 unfreezes the whole model with a differential learning rate;
- **corpus-level WER**. Environment: `sign-language-dnn` (PyTorch 2.6).

## 0 — Setup and configuration (csrl Optuna values)

In [ ]:
import os, pickle, math, time, random
from collections import defaultdict, Counter
from difflib import SequenceMatcher
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt

torch.backends.cudnn.enabled = False

def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(42)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = torch.cuda.is_available()
print('device:', DEVICE, '| torch', torch.__version__, '| amp', AMP)

DATA_DIR  = '/home/ebufi/phoenix_old/MSKA-SLR/data/Phoenix-2014T'
CKPT_PATH = 'tssi75_cslr_best.pt'

CFG = {
    'num_joints': 75,
    # architecture (csrl CONFIG)
    'hidden_dim': 256, 'tcn_blocks': 3, 'num_layers': 3, 'attn_heads': 4,
    'dropout': 0.317638942624273, 'drop_path_rate': 0.1,
    # training
    'batch_size': 16, 'grad_accum': 1,
    'num_epochs': 150, 'early_stopping_patience': 20,
    'phase1_epochs': 15, 'phase1_lr': 0.0001619145621568752,
    'phase2_lr_backbone': 0.00021884694140182422, 'phase2_lr_head': 0.0005038675815696706,
    'weight_decay': 0.0002465667957406484, 'grad_clip': 3.3943208852931304,
    # ctc / decoding
    'ctc_smoothing': 0.10231111042825411, 'prior_beta': 0.32404536014294083, 'beam_width': 25,
    # input
    'max_frames': 400, 'augment': True,
}
print('CFG', CFG)

In [ ]:
import torch
import torch.nn as nn

# Crea una LSTM semplice
rnn = nn.LSTM(10, 20, 2).cuda()
input = torch.randn(5, 3, 10).cuda()
output, (hidden, cell) = rnn(input)
print("GPU con LSTM funzionante!")

GPU con LSTM funzionante!


## 1 — 75-keypoint skeleton: DFS graph + TSSI (per-part normalisation)

In [ ]:
NUM_JOINTS = 75
LH0, RH0 = 33, 54

_pose_edges = [
    (0,1),(1,2),(2,3),(3,7),(0,4),(4,5),(5,6),(6,8),(9,10),
    (11,12),(11,13),(13,15),(12,14),(14,16),
    (15,17),(15,19),(15,21),(17,19),(16,18),(16,20),(16,22),(18,20),
    (11,23),(12,24),(23,24),
    (23,25),(25,27),(27,29),(29,31),(27,31),(24,26),(26,28),(28,30),(30,32),(28,32),
]
_bridge = [(0,11),(0,12),(0,9),(0,10)]
_finger = [
    (0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),(0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),(0,17),(17,18),(18,19),(19,20),
]
_lh = [(a+LH0,b+LH0) for a,b in _finger]
_rh = [(a+RH0,b+RH0) for a,b in _finger]
_wrist_hand = [(15,LH0),(16,RH0)]

_adj = defaultdict(list)
for u,v in _pose_edges + _bridge + _lh + _rh + _wrist_hand:
    _adj[u].append(v); _adj[v].append(u)

def _dfs_order(root=0, n=NUM_JOINTS):
    order, seen = [], set()
    def dfs(u):
        seen.add(u); order.append(u)
        for w in sorted(_adj[u]):
            if w not in seen: dfs(w)
    dfs(root)
    for j in range(n):
        if j not in seen: order.append(j)
    return order

DFS_ORDER = _dfs_order()
assert len(DFS_ORDER) == NUM_JOINTS, len(DFS_ORDER)
JOINT2COL = {j: c for c, j in enumerate(DFS_ORDER)}

_mirror = np.arange(NUM_JOINTS)
for a, b in [(1,4),(2,5),(3,6),(7,8),(9,10),(11,12),(13,14),(15,16),
             (17,18),(19,20),(21,22),(23,24),(25,26),(27,28),(29,30),(31,32)]:
    _mirror[a], _mirror[b] = b, a
for i in range(21):
    _mirror[LH0+i], _mirror[RH0+i] = RH0+i, LH0+i
COL_SWAP = np.array([JOINT2COL[_mirror[DFS_ORDER[c]]] for c in range(NUM_JOINTS)])

PARTS = [(0, 33), (33, 54), (54, 75)]

def generate_tssi_75(kp):
    kp = np.asarray(kp, dtype=np.float32)
    T, J, C = kp.shape
    x = kp[:, :, 0].copy(); y = kp[:, :, 1].copy()
    conf = kp[:, :, 2].copy() if C >= 3 else np.ones((T, J), np.float32)
    x = gaussian_filter1d(x, 1.5, axis=0); y = gaussian_filter1d(y, 1.5, axis=0)
    for s, e in PARTS:
        e = min(e, J)
        for arr in (x, y):
            amin = arr[:, s:e].min(); amax = arr[:, s:e].max()
            arr[:, s:e] = (arr[:, s:e] - amin) / max(amax - amin, 1e-6)
    tssi = np.zeros((3, J, T), np.float32)
    for col, j in enumerate(DFS_ORDER):
        tssi[0, col] = np.clip(x[:, j], 0, 1)
        tssi[1, col] = np.clip(y[:, j], 0, 1)
        tssi[2, col] = np.clip(conf[:, j], 0, 1)
    return tssi

print('DFS 75 ready | in_channels:', NUM_JOINTS*3)

DFS 75 ready | in_channels: 225


## 2 — Vocabulary with gloss merge (as in csrl)

Automatic merge (SequenceMatcher ratio >= 0.85) plus a conservative hyphen-compaction merge,
OOV dev/test tokens mapped to `<unk>`, vocabulary = `['<blank>','<unk>', ...gloss_train]`,
and a log-prior with a low blank probability.

In [ ]:
def build_merge_map(glosses, similarity_threshold=0.85):
    gl = sorted(glosses); mm = {}
    for i, g1 in enumerate(gl):
        if g1 in mm: continue
        for g2 in gl[i+1:]:
            if g2 in mm: continue
            if SequenceMatcher(None, g1, g2).ratio() >= similarity_threshold:
                mm[g2] = g1
    return mm

def apply_safe_hyphen_merges(glosses):
    safe = {}
    for t in sorted(glosses):
        compact = t.replace('-', '')
        if '-' in t and compact in glosses and compact != t:
            safe[t] = compact
    return safe

def _toks(g):
    return str(g).strip().upper().split()

def build_vocab_from_raw(train_raw, dev_raw, test_raw, sim_thr=0.85):
    allg = set()
    for raw in (train_raw, dev_raw, test_raw):
        for s in raw.values(): allg.update(_toks(s['gloss']))
    mm = build_merge_map(allg, sim_thr)
    n_hyphen = 0
    for k, v in apply_safe_hyphen_merges(allg).items():
        if k not in mm: mm[k] = v; n_hyphen += 1
    def mtoks(g): return [mm.get(t, t) for t in _toks(g)]

    traing = set()
    for s in train_raw.values(): traing.update(mtoks(s['gloss']))
    vocab = ['<blank>', '<unk>'] + sorted(traing)
    g2i = {g: i for i, g in enumerate(vocab)}
    i2g = {i: g for g, i in g2i.items()}

    def gloss_to_ids(g, is_train):
        out = []
        for t in mtoks(g):
            if t in g2i and t != '<blank>': out.append(g2i[t])
            elif not is_train: out.append(g2i['<unk>'])
        return out

    cnt = Counter()
    for s in train_raw.values():
        for t in mtoks(s['gloss']):
            if t in g2i: cnt[g2i[t]] += 1
    prior = torch.zeros(len(vocab))
    for idx, c in cnt.items(): prior[idx] = c
    prior[0] = prior[1:].sum() * 0.01          # low blank prior but > 0
    prior = prior / prior.sum()
    log_prior = torch.log(prior + 1e-8)

    print(f'  merge map: {len(mm)} entry ({n_hyphen} hyphen-safe) | vocab: {len(vocab)} classes '
          f'(<blank>=0,<unk>=1, gloss_train={len(traing)})')
    return dict(vocab=vocab, g2i=g2i, i2g=i2g, num_classes=len(vocab),
                gloss_to_ids=gloss_to_ids, log_prior=log_prior, merge_map=mm)

print('vocabulary functions ready')

vocabulary functions ready


## 3 — Dataset (six csrl augmentations) + collate

In [ ]:
def load_pkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

class TSSIDataset(Dataset):
    def __init__(self, raw, gloss_to_ids, is_train, augment=False, max_frames=400):
        self.augment = augment; self.max_frames = max_frames; self.items = []
        for s in raw.values():
            labels = gloss_to_ids(s['gloss'], is_train)
            if labels:
                self.items.append((s['keypoint'], labels))
        print(f'  {len(self.items)} samples | augment={augment}')

    def __len__(self): return len(self.items)

    def _aug(self, tssi):
        C, J, T = tssi.shape
        # 1. temporal Gaussian jitter
        if np.random.rand() < 0.8:
            n = np.random.randn(2, J, T).astype(np.float32)
            n = gaussian_filter1d(n, 3, axis=2) * 0.008
            tssi[:2] = np.clip(tssi[:2] + n, 0, 1)
        # 2. temporal warping
        if np.random.rand() < 0.7:
            sp = np.random.uniform(0.7, 1.3, T).astype(np.float32)
            idx = np.cumsum(sp); idx = (idx - idx[0]) / (idx[-1] - idx[0]) * (T - 1)
            warped = np.zeros_like(tssi)
            for c in range(C):
                for j in range(J):
                    warped[c, j, :] = np.interp(np.arange(T), idx, tssi[c, j, :])
            tssi = warped
        # 3. L/R flip (mirror x + swap left/right hand columns)
        if np.random.rand() < 0.5:
            tssi = tssi.copy(); tssi[0] = 1.0 - tssi[0]; tssi = tssi[:, COL_SWAP, :]
        # 4. scaling
        if np.random.rand() < 0.6:
            tssi[:2] = np.clip(tssi[:2] * np.random.uniform(0.85, 1.15), 0, 1)
        # 5. frame dropout (hold previous)
        if np.random.rand() < 0.5:
            nd = np.random.randint(1, max(2, int(T * 0.05)))
            for i in sorted(np.random.choice(T, nd, replace=False)):
                if i > 0: tssi[:, :, i] = tssi[:, :, i - 1]
        # 6. confidence dropout
        if np.random.rand() < 0.4:
            m = np.random.rand(J, T) < 0.08
            tssi[2, m] = 0.0
        return tssi

    def __getitem__(self, idx):
        kp, labels = self.items[idx]
        kp = np.asarray(kp, dtype=np.float32)
        T = kp.shape[0]
        if T > self.max_frames:
            sel = np.linspace(0, T - 1, self.max_frames).round().astype(int); kp = kp[sel]
        tssi = generate_tssi_75(kp)
        if self.augment: tssi = self._aug(tssi)
        return {'tssi': torch.from_numpy(tssi).float(), 'labels': labels, 'seq_len': tssi.shape[2]}

def collate_ctc(batch):
    Tmax = max(b['tssi'].shape[2] for b in batch)
    xs, il = [], []
    for b in batch:
        t = b['tssi']; xs.append(F.pad(t, (0, Tmax - t.shape[2]))); il.append(t.shape[2])
    x = torch.stack(xs)
    targets = torch.cat([torch.LongTensor(b['labels']) for b in batch])
    return x, targets, torch.LongTensor(il), torch.LongTensor([len(b['labels']) for b in batch])

## 4 — Model: PoseNetworkCTC (TCN + Attention + BiLSTM + CTC + SR-CTC)

In [ ]:
class DropPath(nn.Module):
    def __init__(self, p=0.0): super().__init__(); self.p = p
    def forward(self, x):
        if not self.training or self.p == 0.0: return x
        keep = 1 - self.p; shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        m = (torch.rand(shape, dtype=x.dtype, device=x.device) < keep).float()
        return x * m / keep

class TemporalAttention(nn.Module):
    def __init__(self, dim, heads=4, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(dim); self.drop = nn.Dropout(dropout)
    def forward(self, x):
        a, _ = self.attn(x, x, x); return self.norm(x + self.drop(a))

class TCNBlock(nn.Module):
    def __init__(self, cin, cout, k=3, dilation=1, dropout=0.2, drop_path=0.0):
        super().__init__()
        pad = dilation * (k - 1) // 2
        self.net = nn.Sequential(
            nn.Conv1d(cin, cout, k, padding=pad, dilation=dilation), nn.BatchNorm1d(cout),
            nn.ReLU(True), nn.Dropout(dropout),
            nn.Conv1d(cout, cout, k, padding=pad, dilation=dilation), nn.BatchNorm1d(cout))
        self.skip = (nn.Sequential(nn.Conv1d(cin, cout, 1), nn.BatchNorm1d(cout))
                     if cin != cout else nn.Identity())
        self.dp = DropPath(drop_path)
    def forward(self, x): return F.relu(self.dp(self.net(x)) + self.skip(x))

class PoseNetworkCTC(nn.Module):
    def __init__(self, num_classes, in_channels=225, hidden_dim=256, tcn_blocks=3,
                 lstm_layers=3, dropout=0.3, drop_path_rate=0.1, attn_heads=4):
        super().__init__()
        mid = max(1, tcn_blocks // 2)
        dp = [drop_path_rate * i / max(tcn_blocks - 1, 1) for i in range(tcn_blocks)]
        first = [TCNBlock(in_channels, hidden_dim, 5, 1, dropout, dp[0])]
        for i in range(1, mid):
            first.append(TCNBlock(hidden_dim, hidden_dim, 3, min(2**(i-1), 8), dropout, dp[i]))
        self.tcn_first = nn.Sequential(*first)
        second = [TCNBlock(hidden_dim, hidden_dim, 3, min(2**(i-1), 8), dropout, dp[i])
                  for i in range(mid, tcn_blocks)]
        self.tcn_second = nn.Sequential(*second) if second else nn.Identity()
        self.temporal_attn = TemporalAttention(hidden_dim, attn_heads, dropout * 0.5)
        self.bilstm = nn.LSTM(hidden_dim, hidden_dim, lstm_layers, batch_first=True,
                              bidirectional=True, dropout=dropout if lstm_layers > 1 else 0.0)
        self.norm = nn.LayerNorm(hidden_dim * 2); self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        self.aux_proj = nn.Sequential(nn.Linear(hidden_dim, hidden_dim * 2),
                                      nn.LayerNorm(hidden_dim * 2), nn.Dropout(dropout * 0.5))

    def forward(self, x):
        B, C, J, T = x.shape
        xf = x.reshape(B, C * J, T)
        fm = self.tcn_first(xf).permute(0, 2, 1)
        feat = self.tcn_second(fm.permute(0, 2, 1)).permute(0, 2, 1)
        feat = self.temporal_attn(feat)
        if self.training:
            aux = torch.log_softmax(self.fc(self.aux_proj(fm)), dim=-1)
        feat, _ = self.bilstm(feat)
        feat = self.drop(self.norm(feat))
        main = torch.log_softmax(self.fc(feat), dim=-1)
        return (main, aux) if self.training else main

print('model ready')

model ready


## 5 — CTC + entropy loss, decoding (greedy/beam + prior), corpus-level WER

In [ ]:
class CTCLossWithEntropy(nn.Module):
    def __init__(self, blank=0, entropy_weight=0.05):
        super().__init__()
        self.ctc = nn.CTCLoss(blank=blank, reduction='mean', zero_infinity=True)
        self.ew = entropy_weight
    def forward(self, log_probs_TBC, targets, in_lens, tgt_lens):
        loss = self.ctc(log_probs_TBC, targets, in_lens, tgt_lens)
        if self.ew > 0:
            p = torch.exp(log_probs_TBC).clamp(min=1e-10)
            H = -(p * torch.log(p)).sum(-1).mean() / max(math.log(log_probs_TBC.size(-1)), 1e-8)
            loss = loss + self.ew * H
        return loss

def collapse_ctc(ids):
    out, prev = [], None
    for v in ids:
        v = int(v)
        if v != 0 and v != prev: out.append(v)
        prev = v
    return out

@torch.no_grad()
def greedy_decode(log_probs_TBC, beta=0.0, log_prior=None):
    T, B, C = log_probs_TBC.shape; res = []
    for b in range(B):
        lp = log_probs_TBC[:, b, :]
        if beta > 0 and log_prior is not None: lp = lp - beta * log_prior.unsqueeze(0)
        res.append(collapse_ctc(lp.argmax(-1).tolist()))
    return res

@torch.no_grad()
def beam_decode(log_probs_TBC, beam_width=25, beta=0.3, log_prior=None, topk=None):
    T, B, C = log_probs_TBC.shape; topk = min(topk or beam_width, C); res = []
    for b in range(B):
        lp = log_probs_TBC[:, b, :]
        if beta > 0 and log_prior is not None:
            lp = lp - beta * log_prior.unsqueeze(0)
            lp = lp - torch.logsumexp(lp, dim=1, keepdim=True)
        lp = lp.cpu().numpy()
        beams = {((), None): 0.0}
        for t in range(T):
            cand = np.argpartition(lp[t], -topk)[-topk:]
            if 0 not in cand: cand = np.append(cand, 0)
            nb = {}
            for (pre, last), sc in beams.items():
                for c in cand:
                    c = int(c); nlp = sc + lp[t, c]
                    if c == 0:      key = (pre, last)
                    elif c == last: continue
                    else:           key = (pre + (c,), c)
                    if key not in nb or nb[key] < nlp: nb[key] = nlp
            beams = dict(sorted(nb.items(), key=lambda kv: -kv[1])[:beam_width])
        best = max(beams, key=lambda k: beams[k] / max(len(k[0]), 1) ** 0.7)
        res.append(list(best[0]))
    return res

def compute_wer(refs, hyps):
    tS = tD = tI = tN = 0
    for ref, hyp in zip(refs, hyps):
        r, h = len(ref), len(hyp); tN += r
        if r == 0: tI += h; continue
        dp = [[0]*(h+1) for _ in range(r+1)]
        for i in range(1, r+1): dp[i][0] = i
        for j in range(1, h+1): dp[0][j] = j
        for i in range(1, r+1):
            for j in range(1, h+1):
                dp[i][j] = dp[i-1][j-1] if ref[i-1]==hyp[j-1] else 1+min(dp[i-1][j-1], dp[i-1][j], dp[i][j-1])
        i, j = r, h
        while i > 0 or j > 0:
            if i>0 and j>0 and ref[i-1]==hyp[j-1]: i-=1; j-=1
            elif i>0 and j>0 and dp[i][j]==dp[i-1][j-1]+1: tS+=1; i-=1; j-=1
            elif i>0 and dp[i][j]==dp[i-1][j]+1: tD+=1; i-=1
            else: tI+=1; j-=1
    N = max(tN, 1)
    return (tS+tD+tI)/N, {'S': tS, 'D': tD, 'I': tI, 'N': N}

print('loss / decoding / WER ready')

loss / decoding / WER ready


## 6 — Two-phase training helpers (freeze/unfreeze, optimizers, scheduler)

In [ ]:
def freeze_backbone(m):
    for mod in [m.tcn_first, m.tcn_second, m.temporal_attn]:
        for p in mod.parameters(): p.requires_grad = False
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'  backbone frozen — trainable: {n:,}')

def unfreeze_backbone(m):
    for p in m.parameters(): p.requires_grad = True
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'  backbone unfrozen — trainable: {n:,}')

def make_opt_phase1(m, lr, wd):
    return torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=lr, weight_decay=wd)

def make_opt_phase2(m, lr_b, lr_h, wd):
    groups = [
        {'params': list(m.tcn_first.parameters()),     'lr': lr_b},
        {'params': list(m.tcn_second.parameters()),    'lr': lr_b},
        {'params': list(m.temporal_attn.parameters()), 'lr': lr_b},
        {'params': list(m.bilstm.parameters()),        'lr': lr_h},
        {'params': list(m.norm.parameters()),          'lr': lr_h},
        {'params': list(m.fc.parameters()),            'lr': lr_h},
        {'params': list(m.aux_proj.parameters()),      'lr': lr_h},
    ]
    groups = [g for g in groups if len(g['params']) > 0]
    return torch.optim.AdamW(groups, weight_decay=wd)

def make_sched(opt, num_epochs, steps_per_epoch):
    from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
    ws = 5 * steps_per_epoch; ts = num_epochs * steps_per_epoch
    warm = LinearLR(opt, start_factor=0.1, total_iters=ws)
    cos  = CosineAnnealingLR(opt, T_max=max(ts - ws, 1), eta_min=1e-6)
    return SequentialLR(opt, [warm, cos], milestones=[ws])

print('two-phase helpers ready')

two-phase helpers ready


## 7 — Load data, build vocabulary (merge), instantiate model

In [ ]:
train_raw = load_pkl(os.path.join(DATA_DIR, 'Phoenix-2014T.train'))
dev_raw   = load_pkl(os.path.join(DATA_DIR, 'Phoenix-2014T.dev'))
test_raw  = load_pkl(os.path.join(DATA_DIR, 'Phoenix-2014T.test'))

print('building vocabulary with gloss merge (may take ~1 min\)...')
V = build_vocab_from_raw(train_raw, dev_raw, test_raw)
g2i, i2g = V['g2i'], V['i2g']
NUM_CLASSES = V['num_classes']
gloss_to_ids = V['gloss_to_ids']
LOG_PRIOR = V['log_prior'].to(DEVICE)

train_ds = TSSIDataset(train_raw, gloss_to_ids, is_train=True,  augment=CFG['augment'], max_frames=CFG['max_frames'])
dev_ds   = TSSIDataset(dev_raw,   gloss_to_ids, is_train=False, augment=False,          max_frames=CFG['max_frames'])
test_ds  = TSSIDataset(test_raw,  gloss_to_ids, is_train=False, augment=False,          max_frames=CFG['max_frames'])

train_loader = DataLoader(train_ds, CFG['batch_size'], shuffle=True,  collate_fn=collate_ctc, num_workers=0, drop_last=True)
dev_loader   = DataLoader(dev_ds,   CFG['batch_size'], shuffle=False, collate_fn=collate_ctc, num_workers=0)
test_loader  = DataLoader(test_ds,  CFG['batch_size'], shuffle=False, collate_fn=collate_ctc, num_workers=0)

model = PoseNetworkCTC(num_classes=NUM_CLASSES, in_channels=CFG['num_joints']*3,
                       hidden_dim=CFG['hidden_dim'], tcn_blocks=CFG['tcn_blocks'],
                       lstm_layers=CFG['num_layers'], dropout=CFG['dropout'],
                       drop_path_rate=CFG['drop_path_rate'], attn_heads=CFG['attn_heads']).to(DEVICE)
print('parameters:', round(sum(p.numel() for p in model.parameters())/1e6, 2), 'M')

building vocabulary with gloss merge (may take ~1 min\)...
  merge map: 68 entry (0 hyphen-safe) | vocab: 1022 classes (<blank>=0,<unk>=1, gloss_train=1020)
  7096 samples | augment=True
  519 samples | augment=False
  642 samples | augment=False
parameters: 6.59 M


## 7.5 — Optuna hyperparameter search (optional, ~30% epochs)

In [ ]:
USE_OPTUNA    = True
OPTUNA_TRIALS = 4
OPTUNA_FRAC   = 0.20
PROGRESS_FILE = 'optuna_csrl_progress.json'

if USE_OPTUNA:
    import json as _json
    try:
        import optuna
    except ImportError:
        import subprocess, sys as _s
        subprocess.check_call([_s.executable, '-m', 'pip', 'install', '-q', 'optuna'])
        import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    from optuna.trial import create_trial
    from optuna.distributions import FloatDistribution

    _p1 = max(1, int(CFG['phase1_epochs'] * OPTUNA_FRAC))
    _p2 = max(3, int((CFG['num_epochs'] - CFG['phase1_epochs']) * OPTUNA_FRAC))

    _distributions = {
        'phase1_lr':      FloatDistribution(5e-5, 5e-4, log=True),
        'phase2_lr_head': FloatDistribution(1e-4, 8e-4, log=True),
        'dropout':        FloatDistribution(0.2, 0.45),
        'weight_decay':   FloatDistribution(1e-5, 1e-3, log=True),
    }

    _optuna_log = []
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE) as _f:
            _optuna_log = _json.load(_f)
        print(f'[Optuna] RESUMED — {len(_optuna_log)}/{OPTUNA_TRIALS} trials loaded from {PROGRESS_FILE}')
        for r in _optuna_log:
            print(f'  trial {r["trial"]} | WER {r["wer"]:.2f}% | '
                  f'p1_lr={r["phase1_lr"]:.6f} p2_lr_h={r["phase2_lr_head"]:.6f} '
                  f'drop={r["dropout"]:.4f} wd={r["weight_decay"]:.6f}')
    else:
        print(f'[Optuna] CSRL hyperparameter search (fresh start)')

    print(f'  trials      : {OPTUNA_TRIALS} ({OPTUNA_TRIALS - len(_optuna_log)} remaining)')
    print(f'  phase1      : {_p1} epochs (full: {CFG["phase1_epochs"]})')
    print(f'  phase2      : {_p2} epochs (full: {CFG["num_epochs"] - CFG["phase1_epochs"]})')
    print(f'  searching   : phase1_lr, phase2_lr_head, dropout, weight_decay')
    print(f'  progress    : {PROGRESS_FILE}')
    print('=' * 70)

    def _save_progress():
        with open(PROGRESS_FILE, 'w') as _f:
            _json.dump(_optuna_log, _f, indent=2)

    study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=42))
    for rec in _optuna_log:
        study.add_trial(create_trial(
            params=rec['params'], distributions=_distributions,
            values=[rec['value']],
        ))

    def _trial_wer(trial):
        import copy, gc
        _t0 = time.time()
        cfg_t = copy.deepcopy(CFG)
        cfg_t['phase1_lr']          = trial.suggest_float('phase1_lr', 5e-5, 5e-4, log=True)
        cfg_t['phase2_lr_head']     = trial.suggest_float('phase2_lr_head', 1e-4, 8e-4, log=True)
        cfg_t['phase2_lr_backbone'] = cfg_t['phase2_lr_head'] * 0.5
        cfg_t['dropout']            = trial.suggest_float('dropout', 0.2, 0.45)
        cfg_t['weight_decay']       = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)
        cfg_t['phase1_epochs']      = _p1
        cfg_t['num_epochs']         = _p1 + _p2
        cfg_t['early_stopping_patience'] = max(3, CFG['early_stopping_patience'] // 3)

        print(f'\n--- Trial {trial.number + 1}/{OPTUNA_TRIALS} ---')
        print(f'  phase1_lr      = {cfg_t["phase1_lr"]:.6f}')
        print(f'  phase2_lr_head = {cfg_t["phase2_lr_head"]:.6f}')
        print(f'  phase2_lr_back = {cfg_t["phase2_lr_backbone"]:.6f}')
        print(f'  dropout        = {cfg_t["dropout"]:.4f}')
        print(f'  weight_decay   = {cfg_t["weight_decay"]:.6f}')

        _model = PoseNetworkCTC(num_classes=NUM_CLASSES, in_channels=cfg_t['num_joints']*3,
                                hidden_dim=cfg_t['hidden_dim'], tcn_blocks=cfg_t['tcn_blocks'],
                                lstm_layers=cfg_t['num_layers'], dropout=cfg_t['dropout'],
                                drop_path_rate=cfg_t['drop_path_rate'],
                                attn_heads=cfg_t['attn_heads']).to(DEVICE)
        _crit = CTCLossWithEntropy(blank=0, entropy_weight=cfg_t['ctc_smoothing']).to(DEVICE)
        _scaler = torch.amp.GradScaler('cuda', enabled=AMP)
        def _train_one(opt, sched, m):
            m.train(); tot = 0.0; opt.zero_grad(set_to_none=True)
            for i, (x, tg, il, tl) in enumerate(train_loader):
                x = x.to(DEVICE)
                with torch.amp.autocast('cuda', enabled=AMP):
                    main, aux = m(x)
                    lp = main.permute(1, 0, 2).float()
                    Tout = lp.shape[0]
                    il2 = (il.float() * (Tout / max(x.shape[3], 1))).long().clamp(1, Tout)
                    loss = 0.7 * _crit(lp, tg, il2, tl) + 0.3 * _crit(aux.permute(1,0,2).float(), tg, il2, tl)
                _scaler.scale(loss / cfg_t['grad_accum']).backward()
                if (i + 1) % cfg_t['grad_accum'] == 0 or (i + 1) == len(train_loader):
                    _scaler.unscale_(opt); nn.utils.clip_grad_norm_(m.parameters(), cfg_t['grad_clip'])
                    _scaler.step(opt); _scaler.update(); opt.zero_grad(set_to_none=True)
                sched.step()
                tot += loss.item()
            return tot / len(train_loader)
        @torch.no_grad()
        def _eval(m):
            m.eval(); refs, hyps = [], []
            for x, tg, il, tl in dev_loader:
                x = x.to(DEVICE); lp = m(x).permute(1,0,2)
                dec = greedy_decode(lp, cfg_t['prior_beta'], LOG_PRIOR)
                off = 0
                for b in range(x.size(0)):
                    n = tl[b].item(); refs.append(tg[off:off+n].tolist()); off += n
                hyps.extend(dec)
            return compute_wer(refs, hyps)[0]

        best = 1e9
        freeze_backbone(_model)
        _o1 = make_opt_phase1(_model, cfg_t['phase1_lr'], cfg_t['weight_decay'])
        _s1 = make_sched(_o1, cfg_t['phase1_epochs'], len(train_loader))
        for ep in range(cfg_t['phase1_epochs']):
            loss = _train_one(_o1, _s1, _model)
            wer = _eval(_model)
            best = min(best, wer)
            print(f'    P1 ep {ep+1}/{_p1} | loss {loss:.3f} | dev WER {wer*100:.2f}% | best {best*100:.2f}%')

        unfreeze_backbone(_model)
        _o2 = make_opt_phase2(_model, cfg_t['phase2_lr_backbone'], cfg_t['phase2_lr_head'], cfg_t['weight_decay'])
        _s2 = make_sched(_o2, _p2, len(train_loader))
        for ep in range(_p2):
            loss = _train_one(_o2, _s2, _model)
            wer = _eval(_model)
            best = min(best, wer)
            print(f'    P2 ep {ep+1}/{_p2} | loss {loss:.3f} | dev WER {wer*100:.2f}% | best {best*100:.2f}%')

        elapsed = time.time() - _t0
        _optuna_log.append({
            'trial': trial.number + 1, 'wer': best * 100,
            'phase1_lr': cfg_t['phase1_lr'], 'phase2_lr_head': cfg_t['phase2_lr_head'],
            'dropout': cfg_t['dropout'], 'weight_decay': cfg_t['weight_decay'],
            'time': elapsed,
            'params': trial.params, 'value': float(best),
        })
        _save_progress()
        print(f'  => Trial {trial.number + 1} done | best dev WER = {best*100:.2f}% | {elapsed:.0f}s | SAVED')

        del _model, _o1, _o2, _s1, _s2, _crit, _scaler
        gc.collect(); torch.cuda.empty_cache()
        return float(best)

    _n_remaining = OPTUNA_TRIALS - len(_optuna_log)
    if _n_remaining > 0:
        study.optimize(_trial_wer, n_trials=_n_remaining, show_progress_bar=False)
    else:
        print('All trials already completed.')

    print('\n' + '=' * 70)
    print('[Optuna] SEARCH COMPLETE — summary:')
    print(f'{"#":>3} {"WER%":>7} {"p1_lr":>10} {"p2_lr_h":>10} {"drop":>7} {"wd":>10} {"time":>6}')
    print('-' * 70)
    for r in sorted(_optuna_log, key=lambda x: x['wer']):
        print(f'{r["trial"]:3d} {r["wer"]:7.2f} {r["phase1_lr"]:10.6f} {r["phase2_lr_head"]:10.6f} '
              f'{r["dropout"]:7.4f} {r["weight_decay"]:10.6f} {r["time"]:5.0f}s')
    print(f'\nbest dev WER : {study.best_value*100:.2f}%')
    print(f'best params  : {study.best_params}')
    for k, v in study.best_params.items():
        CFG[k] = v
    CFG['phase2_lr_backbone'] = CFG['phase2_lr_head'] * 0.5
    print('CFG updated with Optuna best.')
else:
    print('Optuna search disabled — using current CFG values.')

[Optuna] RESUMED — 4/4 trials loaded from optuna_csrl_progress.json
  trial 1 | WER 54.40% | p1_lr=0.000118 p2_lr_h=0.000722 drop=0.3830 wd=0.000158
  trial 2 | WER 76.87% | p1_lr=0.000072 p2_lr_h=0.000138 drop=0.2145 wd=0.000540
  trial 3 | WER 51.95% | p1_lr=0.000200 p2_lr_h=0.000436 drop=0.2051 wd=0.000871
  trial 4 | WER 67.72% | p1_lr=0.000340 p2_lr_h=0.000156 drop=0.2455 wd=0.000023
  trials      : 4 (0 remaining)
  phase1      : 3 epochs (full: 15)
  phase2      : 27 epochs (full: 135)
  searching   : phase1_lr, phase2_lr_head, dropout, weight_decay
  progress    : optuna_csrl_progress.json
All trials already completed.

[Optuna] SEARCH COMPLETE — summary:
  #    WER%      p1_lr    p2_lr_h    drop         wd   time
----------------------------------------------------------------------
  3   51.95   0.000200   0.000436  0.2051   0.000871  5717s
  1   54.40   0.000118   0.000722  0.3830   0.000158  5805s
  4   67.72   0.000340   0.000156  0.2455   0.000023  5730s
  2   76.87   0.0

/home/ebufi/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 8 — Two-phase training (Phase 1 freeze → Phase 2 unfreeze + differential LR)

In [ ]:
# ── checkpoint resume config ──
RESUME_PATH = CKPT_PATH.replace('.pt', '_resume.pt')  # checkpoint periodico
SAVE_EVERY  = 5                                        # salva ogni N epoche

def save_resume(path, phase, epoch_in_phase, gep, best, patience, hist,
                opt, sched, scaler):
    torch.save({
        'phase': phase,
        'epoch_in_phase': epoch_in_phase,
        'gep': gep,
        'best': best,
        'patience': patience,
        'hist': hist,
        'model': model.state_dict(),
        'opt': opt.state_dict(),
        'sched': sched.state_dict(),
        'scaler': scaler.state_dict(),
    }, path)
    print(f'   [resume ckpt] saved @ phase={phase} gep={gep}')

def load_resume(path):
    if not os.path.exists(path):
        return None
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    print(f'   [resume ckpt] found: phase={ckpt["phase"]} gep={ckpt["gep"]} best={ckpt["best"]*100:.2f}%')
    return ckpt

# ── tentativo di resume ──
resume = load_resume(RESUME_PATH)

hist = resume['hist'] if resume else {'ep': [], 'phase': [], 'loss': [], 'wer': []}
best = resume['best'] if resume else 1e9
patience = resume['patience'] if resume else 0
gep = resume['gep'] if resume else 0
skip_phase1 = False
start_epoch_p1 = 0
start_epoch_p2 = 0

if resume:
    model.load_state_dict(resume['model'])
    if resume['phase'] == 1:
        start_epoch_p1 = resume['epoch_in_phase']
    elif resume['phase'] == 2:
        skip_phase1 = True
        start_epoch_p2 = resume['epoch_in_phase']

# ---- PHASE 1: frozen backbone, head only ----
if not skip_phase1:
    p1_total = CFG['phase1_epochs']
    remaining_p1 = p1_total - start_epoch_p1
    print('='*60 + f"\nPHASE 1: {p1_total} epochs, FROZEN BACKBONE"
          + (f" (resuming from ep {start_epoch_p1+1})" if start_epoch_p1 > 0 else "")
          + '\n' + '='*60)
    freeze_backbone(model)
    opt1 = make_opt_phase1(model, CFG['phase1_lr'], CFG['weight_decay'])
    sch1 = make_sched(opt1, p1_total, len(train_loader))

    if resume and resume['phase'] == 1:
        opt1.load_state_dict(resume['opt'])
        sch1.load_state_dict(resume['sched'])
        scaler.load_state_dict(resume['scaler'])

    for e in range(start_epoch_p1, p1_total):
        gep += 1; t0 = time.time()
        tr = train_one_epoch(opt1, sch1); wer, det = evaluate(dev_loader)
        hist['ep'].append(gep); hist['phase'].append(1)
        hist['loss'].append(tr); hist['wer'].append(wer*100)
        print(f"[F1] Ep {gep:3d} | loss {tr:.3f} | dev WER {wer*100:.2f}% "
              f"(S{det['S']} D{det['D']} I{det['I']}) | {time.time()-t0:.0f}s")
        if wer < best:
            best = wer
            torch.save({'epoch': gep, 'model': model.state_dict(),
                        'wer': wer, 'cfg': CFG}, CKPT_PATH)
            print(f'   -> best {wer*100:.2f}%')
        if (e + 1) % SAVE_EVERY == 0:
            save_resume(RESUME_PATH, 1, e + 1, gep, best, patience, hist,
                        opt1, sch1, scaler)

# ---- PHASE 2: fully unfrozen, differential LR ----
p2_total = CFG['num_epochs'] - CFG['phase1_epochs']
remaining_p2 = p2_total - start_epoch_p2
print('\n' + '='*60 + f'\nPHASE 2: {p2_total} epochs, UNFREEZE + differential LR'
      + (f" (resuming from ep {start_epoch_p2+1})" if start_epoch_p2 > 0 else "")
      + '\n' + '='*60)
unfreeze_backbone(model)
opt2 = make_opt_phase2(model, CFG['phase2_lr_backbone'], CFG['phase2_lr_head'], CFG['weight_decay'])
sch2 = make_sched(opt2, p2_total, len(train_loader))

if resume and resume['phase'] == 2:
    opt2.load_state_dict(resume['opt'])
    sch2.load_state_dict(resume['sched'])
    scaler.load_state_dict(resume['scaler'])

for e in range(start_epoch_p2, p2_total):
    gep += 1; t0 = time.time()
    tr = train_one_epoch(opt2, sch2); wer, det = evaluate(dev_loader)
    hist['ep'].append(gep); hist['phase'].append(2)
    hist['loss'].append(tr); hist['wer'].append(wer*100)
    print(f"[F2] Ep {gep:3d} | loss {tr:.3f} | dev WER {wer*100:.2f}% "
          f"(S{det['S']} D{det['D']} I{det['I']}) | {time.time()-t0:.0f}s")
    if wer < best:
        best = wer; patience = 0
        torch.save({'epoch': gep, 'model': model.state_dict(),
                    'wer': wer, 'cfg': CFG}, CKPT_PATH)
        print(f'   -> best {wer*100:.2f}%')
    else:
        patience += 1
        if patience >= CFG['early_stopping_patience']:
            print(f'   early stop @ epoch {gep}'); break
    if (e + 1) % SAVE_EVERY == 0:
        save_resume(RESUME_PATH, 2, e + 1, gep, best, patience, hist,
                    opt2, sch2, scaler)

print(f'\nDONE. best dev WER = {best*100:.2f}%')

PHASE 1: 15 epochs, FROZEN BACKBONE
  backbone frozen — trainable: 4,864,510


/home/ebufi/venv/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


[F1] Ep   1 | loss 18.891 | dev WER 100.00% (S0 D3748 I0) | 192s
   -> best 100.00%
[F1] Ep   2 | loss 5.354 | dev WER 99.87% (S0 D3743 I0) | 187s
   -> best 99.87%
[F1] Ep   3 | loss 5.188 | dev WER 99.84% (S1 D3741 I0) | 185s
   -> best 99.84%
[F1] Ep   4 | loss 5.057 | dev WER 99.60% (S10 D3723 I0) | 188s
   -> best 99.60%


/home/ebufi/venv/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


[F1] Ep   5 | loss 4.948 | dev WER 98.96% (S22 D3687 I0) | 187s
   -> best 98.96%
[F1] Ep   6 | loss 4.853 | dev WER 98.11% (S72 D3605 I0) | 183s
   -> best 98.11%
[F1] Ep   7 | loss 4.776 | dev WER 96.53% (S173 D3445 I0) | 184s
   -> best 96.53%
[F1] Ep   8 | loss 4.691 | dev WER 95.94% (S227 D3369 I0) | 189s
   -> best 95.94%
[F1] Ep   9 | loss 4.627 | dev WER 95.52% (S217 D3363 I0) | 193s
   -> best 95.52%
[F1] Ep  10 | loss 4.565 | dev WER 95.33% (S187 D3386 I0) | 193s
   -> best 95.33%
[F1] Ep  11 | loss 4.507 | dev WER 94.93% (S234 D3324 I0) | 193s
   -> best 94.93%
[F1] Ep  12 | loss 4.463 | dev WER 95.01% (S232 D3329 I0) | 193s
[F1] Ep  13 | loss 4.425 | dev WER 95.01% (S231 D3330 I0) | 193s
[F1] Ep  14 | loss 4.404 | dev WER 94.88% (S226 D3330 I0) | 193s
   -> best 94.88%
[F1] Ep  15 | loss 4.399 | dev WER 94.90% (S229 D3328 I0) | 191s

PHASE 2: 135 epochs, UNFREEZE + differential LR
  backbone unfrozen — trainable: 6,593,278
[F2] Ep  16 | loss 4.418 | dev WER 94.82% (S254 D33

## 9 — Plots: loss and WER (with a phase-change marker)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.plot(hist['ep'], hist['loss'], marker='o', ms=3)
ax1.set_title('Train loss (CTC + SR-CTC)'); ax1.set_xlabel('epoch'); ax1.set_ylabel('loss'); ax1.grid(alpha=.3)
if hist['wer']:
    bi = int(np.argmin(hist['wer']))
    ax2.plot(hist['ep'], hist['wer'], marker='o', ms=3, color='crimson')
    ax2.scatter([hist['ep'][bi]], [hist['wer'][bi]], color='k', zorder=5, label=f"best {hist['wer'][bi]:.2f}%")
    if 2 in hist['phase']:
        f2 = hist['ep'][hist['phase'].index(2)]
        ax1.axvline(f2, ls='--', c='gray', alpha=.6); ax2.axvline(f2, ls='--', c='gray', alpha=.6, label='inizio Fase 2')
    ax2.legend()
ax2.set_title('Dev WER (corpus-level)'); ax2.set_xlabel('epoch'); ax2.set_ylabel('WER %'); ax2.grid(alpha=.3)
plt.tight_layout(); plt.savefig('tssi75_curves.png', dpi=110); plt.show()

NameError: name 'plt' is not defined

## 10 — Test set (greedy vs beam)

In [ ]:
ck = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ck['model']); model.to(DEVICE).eval()
print(f"checkpoint epoch {ck['epoch']} | dev WER {ck['wer']*100:.2f}%\n")

wer_g, dg = evaluate(test_loader, use_beam=False)
print(f"TEST greedy      WER {wer_g*100:.2f}%  (S{dg['S']} D{dg['D']} I{dg['I']} N{dg['N']})")
wer_b, db = evaluate(test_loader, use_beam=True)     # beam 25: slower
print(f"TEST beam({CFG['beam_width']})    WER {wer_b*100:.2f}%  (S{db['S']} D{db['D']} I{db['I']} N{db['N']})")
print(f"\n>>> WER to report = {min(wer_g, wer_b)*100:.2f}%")